In [0]:
%pip install transformers==4.53.0 databricks-vectorsearch==0.56 torch
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.3 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Not uninstalling tokenizers at /opt/databricks-environments/databricks-ai/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-114121f7-5779-41b9-9748-4741d244ddf4
    Can't uninstall 'tokenizers'. No files were found to uninstall.
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Not uninstalling transformers at /opt/databricks-environments/databricks-ai/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-114121f7-5779-41b9-9748-4741d244ddf4
    Can't uninstall 'transformers'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. Thi

In [0]:
%sql 
CREATE VOLUME IF NOT EXISTS workspace.default.raw;

In [0]:
%sh
wget -O /Volumes/workspace/default/raw/enwiki-latest-pages-articles.xml https://github.com/MicrosoftLearning/mslearn-databricks/raw/main/data/enwiki-latest-pages-articles.xml

--2026-06-27 15:40:55--  https://github.com/MicrosoftLearning/mslearn-databricks/raw/main/data/enwiki-latest-pages-articles.xml
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/enwiki-latest-pages-articles.xml [following]
--2026-06-27 15:40:55--  https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/enwiki-latest-pages-articles.xml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 25360701 (24M) [text/plain]
Saving to: ‘/Volumes/workspace/default/raw/enwiki-latest-pages-articles.xml’

     0K .......... .......... .......... 

In [0]:

# Read the XML file
raw_df = spark.read.format("xml") \
    .option("rowTag", "page") \
    .load("/Volumes/workspace/default/raw/enwiki-latest-pages-articles.xml")

# Show the DataFrame
raw_df.show(5)

# Print the schema of the DataFrame
raw_df.printSchema()

+---+---+--------------------+--------------------+--------------------+
| id| ns|            redirect|            revision|               title|
+---+---+--------------------+--------------------+--------------------+
| 10|  0|{Computer accessi...|{Restored revisio...| AccessibleComputing|
| 12|  0|                NULL|{Reverted 1 edit ...|           Anarchism|
| 13|  0|{History of Afgha...|{+{{Redirect cate...|  AfghanistanHistory|
| 14|  0|{Geography of Afg...|{+{{Redirect cate...|AfghanistanGeography|
| 15|  0|{Demographics of ...|{+{{Redirect cate...|   AfghanistanPeople|
+---+---+--------------------+--------------------+--------------------+
only showing top 5 rows
root
 |-- id: long (nullable = true)
 |-- ns: long (nullable = true)
 |-- redirect: struct (nullable = true)
 |    |-- _title: string (nullable = true)
 |-- revision: struct (nullable = true)
 |    |-- comment: string (nullable = true)
 |    |-- contributor: struct (nullable = true)
 |    |    |-- id: long (nullable =

In [0]:
from pyspark.sql.functions import col

clean_df = raw_df.select(col("title"), col("revision.text._VALUE").alias("text"))
clean_df = clean_df.na.drop()
clean_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.wiki_pages")
clean_df.show(5)

+--------------------+--------------------+
|               title|                text|
+--------------------+--------------------+
| AccessibleComputing|#REDIRECT [[Compu...|
|           Anarchism|{{Short descripti...|
|  AfghanistanHistory|#REDIRECT [[Histo...|
|AfghanistanGeography|#REDIRECT [[Geogr...|
|   AfghanistanPeople|#REDIRECT [[Demog...|
+--------------------+--------------------+
only showing top 5 rows


In [0]:
%sql
ALTER TABLE workspace.default.wiki_pages SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

client.create_endpoint(
    name="vector_search_endpoint",
    endpoint_type="STANDARD"
)

index = client.create_delta_sync_index(
  endpoint_name="vector_search_endpoint",
  source_table_name="workspace.default.wiki_pages",
  index_name="workspace.default.wiki_index",
  pipeline_type="TRIGGERED",
  primary_key="title",
  embedding_source_column="text",
  embedding_model_endpoint_name="databricks-gte-large-en"
 )

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
results_dict=index.similarity_search(
    query_text="Anthropology fields",
    columns=["title", "text"],
    num_results=1
)

display(results_dict)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'title'}, {'name': 'text'}, {'name': 'score'}]},
 'result': {'row_count': 1,
  'data_array': [['Algeria',
    '{{short description|Country in North Africa}}\n{{About|the country}}\n{{pp-semi-indef}}\n{{pp-move}}\n{{Use dmy dates|date=February 2022}}\n{{Infobox country\n| conventional_long_name = People\'s Democratic Republic of Algeria\n| native_name            = {{native name|ar|الجمهورية الجزائرية الديمقراطية الشعبية}}<br />{{resize|80%|{{transliteration|ar|al-Jumhūriyatu l-Jazāʾiriyatu d-Dīmuqrāṭiyatu sh‑Shaʿbiyah}}}}\n| name                   = \n| common_name            = Algeria\n| image_flag             = Flag of Algeria.svg\n| image_coat             = Emblem of Algeria.svg\n| symbol_type            = [[Emblem of Algeria|Emblem]]\n| national_motto         = {{lang|ar|بِالشَّعْبِ و لِلشَّعْبِ}}<br />"Biš-šaʿb wa liš-šaʿb"<br />"By the people and for the people"<ref name="CONST-AR">{{cite web|url=http://www.el-mouradia.dz/ara

In [0]:
# Convert the dictionary to a DataFrame
results = spark.createDataFrame([results_dict['result']['data_array'][0]])

from transformers import pipeline

# Load the summarization model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", framework="pt")

# Extract the string values from the DataFrame column (serverless-compatible)
text_data = [row["_2"] for row in results.select("_2").collect()]

# Pass the extracted text data to the summarizer function
summary = summarizer(text_data, max_length=512, min_length=100, do_sample=True)

def augment_prompt(query_text):
    context = " ".join([item['summary_text'] for item in summary])
    return f"Query: {query_text}\nContext: {context}"

prompt = augment_prompt("Explain the significance of Anthropology")
print(prompt)

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu129 for torchao version 0.15.0+cu129             Please see https://github.com/pytorch/ao/issues/2919 for more info


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


---------------------------------------------------------------------------
AcceleratorError                          Traceback (most recent call last)
File <command-8576379402727522>, line 13
     10 text_data = [row["_2"] for row in results.select("_2").collect()]
     12 # Pass the extracted text data to the summarizer function
---> 13 summary = summarizer(text_data, max_length=512, min_length=100, do_sample=True)
     15 def augment_prompt(query_text):
     16     context = " ".join([item['summary_text'] for item in summary])

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-114121f7-5779-41b9-9748-4741d244ddf4/lib/python3.12/site-packages/transformers/pipelines/text2text_generation.py:299, in SummarizationPipeline.__call__(self, *args, **kwargs)
    275 def __call__(self, *args, **kwargs):
    276     r"""
    277     Summarize the text(s) given as inputs.
    278 
   (...)
    297           ids of the summary.
    298     """
--> 299     return super().__call__(*args, **kwargs)

F

In [0]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(
    inputs["input_ids"], 
    max_length=300, 
    num_return_sequences=1, 
    repetition_penalty=2.0, 
    top_k=50, 
    top_p=0.95, 
    temperature=0.7,
    do_sample=True
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8576379402727523>, line 6
      3 tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
      4 model = GPT2LMHeadModel.from_pretrained("gpt2")
----> 6 inputs = tokenizer(prompt, return_tensors="pt")
      7 outputs = model.generate(
      8     inputs["input_ids"], 
      9     max_length=300, 
   (...)
     15     do_sample=True
     16 )
     17 response = tokenizer.decode(outputs[0], skip_special_tokens=True)

NameError: name 'prompt' is not defined